# 06 Nested domains: bone and air inside a fat sphere

| | |
|---|---|
| **Purpose** | Build a geometry with domains inside domains using the graph topology, choose a mesh from the wavelength in each material, and solve the nested BEM model. The pattern for tissue models (skull, ribs, organs). |
| **Inputs** | 150 kHz plane wave along +x in water; fat sphere (radius 10 mm) containing a cortical bone sphere (radius 4 mm, centre (-4, 0, 0) mm) and an air sphere (radius 2 mm, centre (4, 0, 0) mm); 4 elements per wavelength on every interface. |
| **Outputs** | Region map and total pressure on z = 0 (`outputs/06_nested_domains_*.png`). |
| **Checks** | The graph has 4 domains and 4 interface nodes (3 spheres plus the one at infinity), with bone and air both inside fat; GMRES converged; field finite; the pressure inside the air sphere is under 1% of the incident amplitude, as expected from the fat to air impedance contrast. |
| **Runtime** | About 40 s full, 28 s CI mode (Docker image, 4 CPUs, 9 Nov 2026). CI mode uses a coarser plotting grid. |
| **Adapted from** | `notebooks/Nested domains.ipynb` (upstream OptimUS). |

## Physics notes

Each interface separates exactly two homogeneous domains, and the graph records which domain contains
which. The mesh on an interface must resolve the *shorter* of the two wavelengths either side of it,
so here the air sphere needs the finest mesh (wavelength 2.3 mm in air against 9.9 mm in water).

The pressure transmitted into air from fat at normal incidence is $T = 2 Z_{air} / (Z_{air} + Z_{fat})$,
about $6 \times 10^{-4}$ (see `05_material_properties`). Air pockets are therefore near-perfect
reflectors, which is why gas in the bowel or lungs blocks therapeutic ultrasound.

> **To expand (Yash):** explain how the graph topology turns into a block system of boundary integral equations, and why the OSRC preconditioner is used on the outer interface.

## Setup

Imports, the `CI_MODE` switch and the output folder.

In [ ]:
NOTEBOOK_NAME = "06_nested_domains"

%matplotlib inline
import json
import os
import time

import numpy as np
import matplotlib.pyplot as plt

import optimus

# CI_MODE uses smaller settings so the notebook finishes in a few minutes in CI.
# Set it from the shell with OPTIMUS_CI_MODE=1, or change it here by hand.
CI_MODE = os.environ.get("OPTIMUS_CI_MODE", "0") == "1"

# Figures and a small JSON file of key results are written here.
OUTPUT_DIR = os.environ.get("OPTIMUS_OUTPUT_DIR", "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

RESULTS = {"notebook": NOTEBOOK_NAME, "ci_mode": CI_MODE,
           "optimus_version": optimus.__version__, "checks": {}}
START_TIME = time.time()

print("OptimUS", optimus.__version__, "| CI mode:", CI_MODE, "| outputs in:", OUTPUT_DIR)

In [ ]:
def check(name, condition, detail):
    """Record a named check and stop the notebook if it fails."""
    RESULTS["checks"][name] = bool(condition)
    print(("PASS  " if condition else "FAIL  ") + name + ": " + detail)
    assert condition, name + " failed: " + detail


def gmres_converged(model):
    """GMRES stops early (before maxiter) only when the tolerance is reached."""
    if hasattr(model, "linear_solve_iteration_count"):
        iterations = model.linear_solve_iteration_count
    else:
        iterations = model.iteration_count
    return iterations, iterations < optimus.global_parameters.linalg.maxiter


def save_figures(figs, stem):
    """Save a list of matplotlib figures as PNG files in OUTPUT_DIR."""
    if not isinstance(figs, (list, tuple)):
        figs = [figs]
    for i, fig in enumerate(figs):
        fig.savefig(os.path.join(OUTPUT_DIR, "%s_%d.png" % (stem, i + 1)), dpi=110,
                    bbox_inches="tight")

## 1. Parameters

In [ ]:
frequency = 150e3
points_per_wavelength = 4
grid_resolution = (80, 60) if CI_MODE else (160, 120)
bounding_box = (-20e-3, 20e-3, -15e-3, 15e-3)

## 2. Source, materials and wavelengths

In [ ]:
source = optimus.source.create_planewave(frequency, direction=(1, 0, 0))
material_ext = optimus.material.load_material("water")
material_fat, material_bone, material_air = optimus.material.load_material(("fat", "bone (cortical)", "air"))
materials = [material_ext, material_fat, material_bone, material_air]
wavelengths = [m.compute_wavelength(frequency) for m in materials]
for m, w in zip(materials, wavelengths):
    print("Wavelength in %-16s %.2f mm" % (m.name + ":", w * 1e3))

## 3. Geometry as a graph

In [ ]:
graph = optimus.geometry.Graph("three_nested_scatterers")
graph.create_exterior_domain(material_ext, source)

meshwidth_fat = min(wavelengths[0], wavelengths[1]) / points_per_wavelength
meshwidth_bone = min(wavelengths[1], wavelengths[2]) / points_per_wavelength
meshwidth_air = min(wavelengths[1], wavelengths[3]) / points_per_wavelength

sphere_fat = optimus.geometry.shapes.Sphere(radius=10e-3, origin=(0, 0, 0),
                                            element_size=meshwidth_fat, label="fat")
sphere_bone = optimus.geometry.shapes.Sphere(radius=4e-3, origin=(-4e-3, 0, 0),
                                             element_size=meshwidth_bone, label="bone")
sphere_air = optimus.geometry.shapes.Sphere(radius=2e-3, origin=(4e-3, 0, 0),
                                            element_size=meshwidth_air, label="air")

graph.create_interface(sphere_fat, 0, material_fat)    # inside domain 0 (water)
graph.create_interface(sphere_bone, 1, material_bone)  # inside domain 1 (fat)
graph.create_interface(sphere_air, 1, material_air)    # inside domain 1 (fat)
graph.print_graph_information()
RESULTS["number_of_vertices"] = int(sum(g.number_of_vertices() for g in (sphere_fat, sphere_bone, sphere_air)))

## 4. Model, region check and solve

PMCHWT formulation with the OSRC preconditioner on the large outer interface and mass matrix
preconditioning on the two small ones. Plotting the regions first is a cheap way to catch a wrongly
nested geometry before the solve.

In [ ]:
model = optimus.model.create_nested_model(graph, frequency, "pmchwt", ["osrc", "mass", "mass"])
plane = optimus.postprocess.VisualisePlane(model)
plane.create_computational_grid(resolution=grid_resolution, bounding_box=bounding_box)
fig = optimus.postprocess.plot_regions(plane)
save_figures(fig, NOTEBOOK_NAME + "_regions")

In [ ]:
t0 = time.time()
model.solve()
RESULTS["solve_time_s"] = round(time.time() - t0, 1)
iterations, converged = gmres_converged(model)
RESULTS["gmres_iterations"] = int(iterations)
print("GMRES iterations:", iterations, "| solve time %.1f s" % RESULTS["solve_time_s"])

## 5. Field on the plane z = 0

In [ ]:
plane.compute_fields()
figs = optimus.postprocess.plot_pressure_field(plane)
save_figures(figs, NOTEBOOK_NAME + "_total")

## 6. Checks

Measured on 9 Nov 2026: the largest |p| inside the air sphere was 0.0022 (CI) and 0.0025 (full), with incident amplitude 1,
against 1.2 in the surrounding water. The 1% limit leaves a wide margin while still failing if the air
sphere were treated as tissue.

In [ ]:
fat_children = graph.subdomain_nodes[1].child_interfaces_ids
check("graph_topology",
      graph.number_interface_nodes() == 4 and graph.number_subdomain_nodes() == 4
      and len(fat_children) == 2,
      "%d interface nodes (3 spheres + the one at infinity), %d subdomains, "
      "fat domain contains interfaces %s"
      % (graph.number_interface_nodes(), graph.number_subdomain_nodes(), str(fat_children)))
check("gmres_converged", converged, "%d iterations" % iterations)

total = plane.field.total_field
check("field_is_finite", np.all(np.isfinite(total)),
      "%d NaN or inf values on the plane" % np.sum(~np.isfinite(total)))

points = plane.points
in_air = np.linalg.norm(points - np.array([[4e-3], [0], [0]]), axis=0) < 0.9 * 2e-3
p_air = np.max(np.abs(total[in_air]))
z_fat = material_fat.density * material_fat.speed_of_sound
z_air = material_air.density * material_air.speed_of_sound
RESULTS["max_pressure_in_air"] = float(p_air)
RESULTS["transmission_fat_to_air"] = float(2 * z_air / (z_air + z_fat))
check("air_blocks_sound", p_air < 0.01,
      "max |p| in air = %.4f at %d points (T fat to air = %.1e), limit 0.01"
      % (p_air, in_air.sum(), 2 * z_air / (z_air + z_fat)))

## Summary

Key numbers and check results, also saved as JSON in `OUTPUT_DIR` so CI and the assistant can read them.

In [ ]:
RESULTS["runtime_s"] = round(time.time() - START_TIME, 1)
with open(os.path.join(OUTPUT_DIR, NOTEBOOK_NAME + "_results.json"), "w") as f:
    json.dump(RESULTS, f, indent=2, sort_keys=True)
print(json.dumps(RESULTS, indent=2, sort_keys=True))